In [10]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
from google.colab import files

class Config:
    DATASET_CHOICE = 'fashion'
    EPOCHS = 50
    BATCH_SIZE = 128
    NOISE_DIM = 100
    LEARNING_RATE_G = 0.0004
    LEARNING_RATE_D = 0.0001
    SAVE_INTERVAL = 5

    SAMPLE_DIR = 'generated_samples/'
    FINAL_DIR = 'final_generated_images/'

if os.path.exists(Config.SAMPLE_DIR):
    shutil.rmtree(Config.SAMPLE_DIR)
if os.path.exists(Config.FINAL_DIR):
    shutil.rmtree(Config.FINAL_DIR)

os.makedirs(Config.SAMPLE_DIR)
os.makedirs(Config.FINAL_DIR)

print(f"Configuration loaded. Dataset: {Config.DATASET_CHOICE}")

Configuration loaded. Dataset: fashion


In [11]:
def load_data(choice):
    print(f"Loading {choice} dataset...")
    if choice == 'mnist':
        (x_train, y_train), (_, _) = datasets.mnist.load_data()
    elif choice == 'fashion':
        (x_train, y_train), (_, _) = datasets.fashion_mnist.load_data()

    # Normalize to [-1, 1]
    x_train = (x_train.astype(np.float32) - 127.5) / 127.5
    x_train = np.expand_dims(x_train, axis=3)
    return x_train, y_train

X_train, Y_train_labels = load_data(Config.DATASET_CHOICE)
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(Config.BATCH_SIZE)
print("Data ready.")

Loading fashion dataset...
Data ready.


In [12]:
def build_generator(noise_dim):
    """Generator: Optimized to prevent Mode Collapse (Black Images)"""
    model = models.Sequential(name="Generator")

    # Dense Layer
    model.add(layers.Dense(7 * 7 * 128, input_dim=noise_dim))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())
    model.add(layers.Reshape((7, 7, 128)))

    # Upsample to 14x14
    model.add(layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # Upsample to 28x28
    model.add(layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # Output Layer
    model.add(layers.Conv2D(1, (7, 7), activation='tanh', padding='same'))
    return model

def build_discriminator(img_shape=(28, 28, 1)):
    """Discriminator"""
    model = models.Sequential(name="Discriminator")
    model.add(layers.Conv2D(64, (3, 3), strides=(2, 2), padding='same', input_shape=img_shape))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (3, 3), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU(alpha=0.2))
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

generator = build_generator(Config.NOISE_DIM)
discriminator = build_discriminator()
print("Models initialized with robust architecture.")

Models initialized with robust architecture.


In [13]:
g_optimizer = tf.keras.optimizers.Adam(Config.LEARNING_RATE_G, beta_1=0.5)
d_optimizer = tf.keras.optimizers.Adam(Config.LEARNING_RATE_D, beta_1=0.5)
cross_entropy = tf.keras.losses.BinaryCrossentropy()

@tf.function
def train_step(images):
    noise = tf.random.normal([Config.BATCH_SIZE, Config.NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)


        real_images_noisy = images + tf.random.normal(shape=tf.shape(images), stddev=0.1)

        real_output = discriminator(real_images_noisy, training=True)
        fake_output = discriminator(generated_images, training=True)

        real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
        fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        d_loss = real_loss + fake_loss

        g_loss = cross_entropy(tf.ones_like(fake_output), fake_output)

    gradients_of_generator = gen_tape.gradient(g_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(d_loss, discriminator.trainable_variables)

    g_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    d_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return d_loss, g_loss

In [14]:
def save_plot(epoch, generator, examples=25):
    """Saves a grid of generated images (Fixed Normalization)"""
    noise = tf.random.normal([examples, Config.NOISE_DIM])
    gen_imgs = generator(noise, training=False)

    # Scale from [-1, 1] to [0, 1]
    gen_imgs = (gen_imgs + 1) / 2.0

    fig, axs = plt.subplots(5, 5, figsize=(5, 5))
    cnt = 0
    for i in range(5):
        for j in range(5):
            axs[i, j].imshow(gen_imgs[cnt, :, :, 0], cmap='gray')
            axs[i, j].axis('off')
            cnt += 1

    path = os.path.join(Config.SAMPLE_DIR, f"epoch_{epoch:02d}.png")
    plt.savefig(path)
    plt.close()

print(f"Starting training for {Config.EPOCHS} epochs...")

for epoch in range(1, Config.EPOCHS + 1):
    d_losses, g_losses = [], []
    for batch in dataset:
        d, g = train_step(batch)
        d_losses.append(d)
        g_losses.append(g)

    print(f"Epoch {epoch}/{Config.EPOCHS} | D_loss: {np.mean(d_losses):.4f} | G_loss: {np.mean(g_losses):.4f}")

    if epoch % Config.SAVE_INTERVAL == 0 or epoch == Config.EPOCHS:
        save_plot(epoch, generator)

print("Training Complete.")

Starting training for 50 epochs...
Epoch 1/50 | D_loss: 1.3871 | G_loss: 0.7702
Epoch 2/50 | D_loss: 1.3910 | G_loss: 0.7747
Epoch 3/50 | D_loss: 1.3811 | G_loss: 0.7935
Epoch 4/50 | D_loss: 1.3787 | G_loss: 0.7944
Epoch 5/50 | D_loss: 1.3762 | G_loss: 0.7960
Epoch 6/50 | D_loss: 1.3731 | G_loss: 0.7990
Epoch 7/50 | D_loss: 1.3644 | G_loss: 0.8058
Epoch 8/50 | D_loss: 1.3433 | G_loss: 0.8205
Epoch 9/50 | D_loss: 1.2783 | G_loss: 0.8722
Epoch 10/50 | D_loss: 1.1760 | G_loss: 0.9682
Epoch 11/50 | D_loss: 1.0551 | G_loss: 1.1071
Epoch 12/50 | D_loss: 0.8983 | G_loss: 1.3513
Epoch 13/50 | D_loss: 0.7822 | G_loss: 1.6157
Epoch 14/50 | D_loss: 0.7051 | G_loss: 1.8604
Epoch 15/50 | D_loss: 0.6562 | G_loss: 2.0759
Epoch 16/50 | D_loss: 0.6238 | G_loss: 2.2466
Epoch 17/50 | D_loss: 0.6022 | G_loss: 2.3947
Epoch 18/50 | D_loss: 0.5893 | G_loss: 2.4898
Epoch 19/50 | D_loss: 0.5745 | G_loss: 2.5905
Epoch 20/50 | D_loss: 0.5645 | G_loss: 2.6664
Epoch 21/50 | D_loss: 0.5551 | G_loss: 2.7299
Epoch 22

In [15]:
print("Generating final outputs...")

final_noise = tf.random.normal([100, Config.NOISE_DIM])
final_imgs = generator(final_noise, training=False)

final_imgs_unscaled = ((final_imgs * 0.5 + 0.5) * 255).numpy().astype(np.uint8)
for i in range(100):
    img_path = os.path.join(Config.FINAL_DIR, f"synthetic_{i+1:03d}.png")
    tf.keras.utils.save_img(img_path, final_imgs_unscaled[i])


print("Training classifier for evaluation...")
classifier = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])
classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
classifier.fit(X_train, Y_train_labels, epochs=1, batch_size=64, verbose=0)

preds = classifier.predict(final_imgs)
pred_labels = np.argmax(preds, axis=1)

print("\n" + "="*40)
print(" Output 4: Labels of Generated Images")
print("="*40)
unique, counts = np.unique(pred_labels, return_counts=True)
labels_map = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

for u, c in zip(unique, counts):
    name = labels_map[u] if Config.DATASET_CHOICE == 'fashion' else str(u)
    print(f"{name}: {c}")

print("\nZipping and downloading results...")
shutil.make_archive('generated_samples', 'zip', Config.SAMPLE_DIR)
shutil.make_archive('final_generated_images', 'zip', Config.FINAL_DIR)

files.download('generated_samples.zip')
files.download('final_generated_images.zip')

Generating final outputs...
Training classifier for evaluation...


1/4 ━━━━━━━━━━━━━━━━━━━━ 1s 363ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step

 Output 4: Labels of Generated Images
Pullover: 11
Shirt: 34
Bag: 40
Boot: 15

Zipping and downloading results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>